# TransNeXt Fine-tuning on ISIC 2018 Task 3 
 
---
 
## Install
 
---
 
### 🛠️ Environment Setup
 
Kaggle's default runtime runs Python 3.7, but TransNeXt requires Python 3.8+ due to newer typing syntax. The first step is to install **Python 3.8** and create a dedicated **virtual environment** (`venv`) so all libraries are installed at the correct version:
 
- `python3.8-dev` — header files required to build CUDA extensions (swattention)
- `virtualenv` — manages an isolated environment separate from Kaggle's default conda
 
After creating the venv, `sys.path` and the `PATH`/`VIRTUAL_ENV` environment variables are patched so the Python runtime picks up the correct libraries from inside the venv.
 
---

In [ ]:
!apt-get update
!apt-get install python3.8-dev -y

In [5]:
!python3.8 --version
!dpkg -l | grep python3.8-dev

Python 3.8.10
ii  libpython3.8-dev:amd64            3.8.10-0ubuntu1~20.04.18          amd64        Header files and a static library for Python (v3.8)
ii  python3.8-dev                     3.8.10-0ubuntu1~20.04.18          amd64        Header files and a static library for Python (v3.8)


In [6]:
!pip install virtualenv

%cd /kaggle/working
!virtualenv venv -p $(which python3.8)
# !virtualenv myenv

/kaggle/working
created virtual environment CPython3.8.10.final.0-64 in 894ms
  creator CPython3Posix(dest=/kaggle/working/venv, clear=False, no_vcs_ignore=False, global=False)
  seeder FromAppData(download=False, pip=bundle, setuptools=bundle, wheel=bundle, via=copy, app_data_dir=/root/.local/share/virtualenv)
    added seed packages: MarkupSafe==2.1.5, PyYAML==6.0.3, addict==2.4.0, certifi==2022.12.7, charset_normalizer==2.1.1, cmake==3.25.0, filelock==3.16.1, idna==3.4, jinja2==3.1.6, lit==15.0.7, mmcv_full==1.7.2, mpmath==1.3.0, networkx==3.1, numpy==1.24.4, opencv_python==4.13.0.92, packaging==26.0, pillow==10.4.0, pip==22.2.2, platformdirs==4.3.6, pyngrok==7.2.12, requests==2.28.1, setuptools==65.3.0, sympy==1.13.3, timm==0.5.4, tomli==2.4.1, torch==2.0.1+cu118, torchaudio==2.0.2+cu118, torchvision==0.15.2+cu118, triton==2.0.0, typing_extensions==4.12.2, urllib3==1.26.13, wheel==0.37.1, yapf==0.43.0
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerSh

In [7]:
import sys
sys.path.insert(0, '/kaggle/working/venv/lib/python3.8/site-packages')

import os
os.environ['VIRTUAL_ENV'] = '/kaggle/working/venv'
os.environ['PATH'] = '/kaggle/working/venv/bin:' + os.environ['PATH']

### 📦 Installed Library Stack
 
| Library | Version | Notes |
|---|---|---|
| Python | 3.8.10 | Installed via `python3.8-dev` |
| PyTorch | 2.0.1 | Built for CUDA 11.8 (`cu118`) |
| torchvision | 0.15.2 | Synchronized with PyTorch |
| mmcv-full | 1.7.2 | Required for TransNeXt's Aggregated Attention module |
| timm | 0.5.4 | Image model zoo, used for augmentation & scheduler |
 
> **Note:** mmcv-full is installed from a pre-built wheel for `cu117/torch2.0.0` since no `cu118` wheel exists — it is fully runtime-compatible.
 

In [ ]:
!pip install torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118 --no-cache-dir

In [ ]:
!pip install -q mmcv-full==1.7.2 -f https://download.openmmlab.com/mmcv/dist/cu117/torch2.0.0/index.html

In [ ]:
!pip show mmcv-full
!pip show torch
!python3.8 --version
!python --version
!pip --version
!nvcc --version

In [ ]:
!git clone https://github.com/DaiShiResearch/TransNeXt.git

In [8]:
%%writefile /kaggle/working/TransNeXt/classification/requirements.txt
# torch==2.0.1
# torchvision==0.15.2
timm==0.5.4
# mmcv==1.4.3


Overwriting /kaggle/working/TransNeXt/classification/requirements.txt


In [9]:
%cd /kaggle/working/TransNeXt/classification 
!pip install -r requirements.txt

/kaggle/working/TransNeXt/classification

[notice] A new release of pip available: 22.2.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [10]:
%cd /kaggle/working/TransNeXt/swattention_extension
!pip install -e .

/kaggle/working/TransNeXt/swattention_extension
Obtaining file:///kaggle/working/TransNeXt/swattention_extension
  Preparing metadata (setup.py) ... done
  Attempting uninstall: swattention
    Found existing installation: swattention 1.0
    Uninstalling swattention-1.0:
      Successfully uninstalled swattention-1.0
  Running setup.py develop for swattention

[notice] A new release of pip available: 22.2.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


### ⚠️ Fixing CUDA Runtime Issues on Kaggle
 
When building `swattention_extension` (TransNeXt's custom CUDA op), the linker cannot find `libcuda.so` because Kaggle places it under `/usr/local/nvidia/lib64/` instead of `/usr/lib/` (the default linker search path).
 
**Fix:** Manually create symlinks:
```
/usr/lib/libcuda.so   →  /usr/local/nvidia/lib64/libcuda.so
/usr/lib/libcuda.so.1 →  /usr/local/nvidia/lib64/libcuda.so.1
```
 
Additionally, the `ptxas` binary (the CUDA PTX compiler used by Triton/PyTorch compiler) needs executable permissions (`chmod +x`). After these fixes, the `swattention` CUDA extension loads successfully on every subsequent run.
 

In [11]:
!find / -name "libcuda.so*" 2>/dev/null

/usr/local/nvidia/lib64/libcuda.so.1
/usr/local/nvidia/lib64/libcuda.so.580.105.08
/usr/local/nvidia/lib64/libcuda.so
/usr/local/cuda-11.0/compat/libcuda.so.1
/usr/local/cuda-11.0/compat/libcuda.so
/usr/local/cuda-11.0/compat/libcuda.so.450.191.01


In [12]:
# Tạo symlink trỏ đúng vào chỗ linker tìm
!ln -sf /usr/local/nvidia/lib64/libcuda.so /usr/lib/libcuda.so
!ln -sf /usr/local/nvidia/lib64/libcuda.so.1 /usr/lib/libcuda.so.1

# Refresh linker cache
!ldconfig

In [13]:
!ls -la /usr/lib/libcuda.so
!ls -la /usr/lib/libcuda.so.1

lrwxrwxrwx 1 root root 34 Apr  5 13:25 /usr/lib/libcuda.so -> /usr/local/nvidia/lib64/libcuda.so
lrwxrwxrwx 1 root root 36 Apr  5 13:25 /usr/lib/libcuda.so.1 -> /usr/local/nvidia/lib64/libcuda.so.1


In [14]:
!chmod +x /kaggle/working/venv/lib/python3.8/site-packages/triton/third_party/cuda/bin/ptxas
!ls -la /kaggle/working/venv/lib/python3.8/site-packages/triton/third_party/cuda/bin/ptxas

-rwxr-xr-x 1 root root 19672208 Apr  5 12:46 /kaggle/working/venv/lib/python3.8/site-packages/triton/third_party/cuda/bin/ptxas


### 📊 ISIC 2018 Task 3 — Dataset Distribution
 
The dataset contains **7 skin lesion classes**:
 
| Label | Train | Val | Test | Description |
|---|---|---|---|---|
| NV | 6705 | 123 | 909 | Melanocytic nevi (benign moles) |
| MEL | 1113 | 21 | 171 | Melanoma (malignant) |
| BKL | 1099 | 22 | 217 | Benign keratosis |
| BCC | 514 | 15 | 93 | Basal cell carcinoma |
| AKIEC | 327 | 8 | 43 | Actinic keratosis |
| VASC | 142 | 3 | 35 | Vascular lesions |
| DF | 115 | 1 | 44 | Dermatofibroma |
| **Total** | **10,015** | **193** | **1,512** | |
 
> **⚠️ Severe class imbalance:** NV accounts for ~67% of the training set, while DF has only 115 images. This is a typical challenge in medical imaging — the model is prone to bias toward the majority class.
 
Images are organized in `class/image.jpg` folder structure to be compatible with torchvision's `ImageFolder`. The CSV ground-truth labels are parsed and images are copied into the correct subfolders for each split (train / val / test).
 

## Data process

In [15]:
import os
import pandas as pd
import shutil
from tqdm import tqdm

# ================== CONFIG ==================
BASE_OUT = "/kaggle/tmp/isic_2018_task3"

DATA_PATHS = {
    "train": {
        "img_dir": "/kaggle/input/datasets/dinhnhatky2003/isic-2018/ISIC2018_Task3_Training_Input/ISIC2018_Task3_Training_Input",
        "csv": "/kaggle/input/datasets/dinhnhatky2003/isic-2018/ISIC2018_Task3_Training_GroundTruth/ISIC2018_Task3_Training_GroundTruth/ISIC2018_Task3_Training_GroundTruth.csv"
    },
    "val": {
        "img_dir": "/kaggle/input/datasets/dinhnhatky2003/isic-2018/ISIC2018_Task3_Validation_Input/ISIC2018_Task3_Validation_Input",
        "csv": "/kaggle/input/datasets/dinhnhatky2003/isic-2018/ISIC2018_Task3_Validation_GroundTruth/ISIC2018_Task3_Validation_GroundTruth/ISIC2018_Task3_Validation_GroundTruth.csv"
    },
    "test": {
        "img_dir": "/kaggle/input/datasets/dinhnhatky2003/isic-2018/ISIC2018_Task3_Test_Input/ISIC2018_Task3_Test_Input",
        "csv": "/kaggle/input/datasets/dinhnhatky2003/isic-2018/ISIC2018_Task3_Test_GroundTruth/ISIC2018_Task3_Test_GroundTruth/ISIC2018_Task3_Test_GroundTruth.csv"
    }
}

CLASSES = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]

# ============================================

def prepare_dirs():
    for split in ["train", "val", "test"]:
        for cls in CLASSES:
            path = os.path.join(BASE_OUT, split, cls)
            os.makedirs(path, exist_ok=True)

def process_split(split_name, img_dir, csv_path):
    df = pd.read_csv(csv_path)

    print(f"\nProcessing {split_name}...")

    for _, row in tqdm(df.iterrows(), total=len(df)):
        image_id = row["image"]

        # find label (one-hot → class)
        label = None
        for cls in CLASSES:
            if row[cls] == 1:
                label = cls
                break

        if label is None:
            continue

        src = os.path.join(img_dir, image_id + ".jpg")
        dst = os.path.join(BASE_OUT, split_name, label, image_id + ".jpg")

        if os.path.exists(src):
            shutil.copy(src, dst)
        else:
            print(f"Missing: {src}")

# ================= RUN =================

prepare_dirs()

for split, info in DATA_PATHS.items():
    process_split(split, info["img_dir"], info["csv"])

print("\nDone!")


Processing train...


100%|██████████| 10015/10015 [01:49<00:00, 91.75it/s]



Processing val...


100%|██████████| 193/193 [00:02<00:00, 88.70it/s]



Processing test...


100%|██████████| 1512/1512 [00:15<00:00, 95.73it/s] 


Done!


In [16]:
!ls /kaggle/tmp/isic_2018_task3
!ls /kaggle/tmp/isic_2018_task3/train
!ls /kaggle/tmp/isic_2018_task3/val
!ls /kaggle/tmp/isic_2018_task3/test

import os
labels = ["AKIEC",  "BCC",  "BKL",  "DF",  "MEL",  "NV",  "VASC"]

for slit in ["train", "val", "test"]:
    print("Split", slit)
    for label in labels:
        print(label + ": ", len(os.listdir(os.path.join("/kaggle/tmp/isic_2018_task3", slit, label))))
    print()

test  train  val
AKIEC  BCC  BKL  DF  MEL  NV  VASC
AKIEC  BCC  BKL  DF  MEL  NV  VASC
AKIEC  BCC  BKL  DF  MEL  NV  VASC
Split train
AKIEC:  327
BCC:  514
BKL:  1099
DF:  115
MEL:  1113
NV:  6705
VASC:  142

Split val
AKIEC:  8
BCC:  15
BKL:  22
DF:  1
MEL:  21
NV:  123
VASC:  3

Split test
AKIEC:  43
BCC:  93
BKL:  217
DF:  44
MEL:  171
NV:  909
VASC:  35



In [33]:
%%writefile /kaggle/working/TransNeXt/classification/datasets.py
# Copyright (c) 2015-present, Facebook, Inc.
# All rights reserved.
import os
import json

from torchvision import datasets, transforms
from torchvision.datasets.folder import ImageFolder, default_loader

from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
from timm.data import create_transform
from mcloader import ClassificationDataset


class INatDataset(ImageFolder):
    def __init__(self, root, train=True, year=2018, transform=None, target_transform=None,
                 category='name', loader=default_loader):
        self.transform = transform
        self.loader = loader
        self.target_transform = target_transform
        self.year = year
        # assert category in ['kingdom','phylum','class','order','supercategory','family','genus','name']
        path_json = os.path.join(root, f'{"train" if train else "val"}{year}.json')
        with open(path_json) as json_file:
            data = json.load(json_file)

        with open(os.path.join(root, 'categories.json')) as json_file:
            data_catg = json.load(json_file)

        path_json_for_targeter = os.path.join(root, f"train{year}.json")

        with open(path_json_for_targeter) as json_file:
            data_for_targeter = json.load(json_file)

        targeter = {}
        indexer = 0
        for elem in data_for_targeter['annotations']:
            king = []
            king.append(data_catg[int(elem['category_id'])][category])
            if king[0] not in targeter.keys():
                targeter[king[0]] = indexer
                indexer += 1
        self.nb_classes = len(targeter)

        self.samples = []
        for elem in data['images']:
            cut = elem['file_name'].split('/')
            target_current = int(cut[2])
            path_current = os.path.join(root, cut[0], cut[2], cut[3])

            categors = data_catg[target_current]
            target_current_true = targeter[categors[category]]
            self.samples.append((path_current, target_current_true))

    # __getitem__ and __len__ inherited from ImageFolder

def build_dataset(is_train, args):
    transform = build_transform(is_train, args)
    eval_split = getattr(args, 'eval_split', 'val')  # backward-compatible
    split = 'train' if is_train else eval_split

    if args.data_set == 'CIFAR':
        dataset = datasets.CIFAR100(args.data_path, train=is_train, transform=transform)
        nb_classes = 100

    elif args.data_set == 'IMNET':
        if not args.use_mcloader:
            root = os.path.join(args.data_path, split)
            dataset = datasets.ImageFolder(root, transform=transform)
        else:
            dataset = ClassificationDataset(split, pipeline=transform)
        nb_classes = 1000

    elif args.data_set == 'INAT':
        dataset = INatDataset(args.data_path, train=is_train, year=2018,
                              category=args.inat_category, transform=transform)
        nb_classes = dataset.nb_classes

    elif args.data_set == 'INAT19':
        dataset = INatDataset(args.data_path, train=is_train, year=2019,
                              category=args.inat_category, transform=transform)
        nb_classes = dataset.nb_classes

    elif args.data_set == 'ISIC_2018':
        root = os.path.join(args.data_path, split)
        dataset = datasets.ImageFolder(root, transform=transform)

        expected_classes = ['AKIEC', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'VASC']
        found_classes = sorted(dataset.class_to_idx.keys())
        if found_classes != sorted(expected_classes):
            raise ValueError(
                f"ISIC_2018 classes mismatch. "
                f"Found: {found_classes}, "
                f"Expected: {sorted(expected_classes)}"
            )
        nb_classes = 7
    else:
        raise ValueError(f"Unsupported dataset: {args.data_set}")

    return dataset, nb_classes


# def build_dataset(is_train, args):
#     transform = build_transform(is_train, args)

#     if args.data_set == 'CIFAR':
#         dataset = datasets.CIFAR100(args.data_path, train=is_train, transform=transform)
#         nb_classes = 100
#     elif args.data_set == 'IMNET':
#         if not args.use_mcloader:
#             root = os.path.join(args.data_path, 'train' if is_train else 'val')
#             dataset = datasets.ImageFolder(root, transform=transform)
#         else:
#             dataset = ClassificationDataset(
#                 'train' if is_train else 'val',
#                 pipeline=transform
#             )
#         nb_classes = 1000
#     elif args.data_set == 'INAT':
#         dataset = INatDataset(args.data_path, train=is_train, year=2018,
#                               category=args.inat_category, transform=transform)
#         nb_classes = dataset.nb_classes
#     elif args.data_set == 'INAT19':
#         dataset = INatDataset(args.data_path, train=is_train, year=2019,
#                               category=args.inat_category, transform=transform)
#         nb_classes = dataset.nb_classes
#     elif args.data_set == 'ISIC_2018':
#         root = os.path.join(args.data_path, 'train' if is_train else 'val')
#         dataset = datasets.ImageFolder(root, transform=transform)

#         expected_classes = ['AKIEC', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'VASC']
#         found_classes = sorted(dataset.class_to_idx.keys())
#         if found_classes != sorted(expected_classes):
#             raise ValueError(
#                 f"ISIC_2018 classes mismatch. "
#                 f"Found: {found_classes}, "
#                 f"Expected: {sorted(expected_classes)}"
#             )
#         nb_classes = 7
#     else:
#         raise ValueError(f"Unsupported dataset: {args.data_set}")

#     return dataset, nb_classes


def build_transform(is_train, args):
    resize_im = args.input_size > 32
    if is_train:
        # this should always dispatch to transforms_imagenet_train
        transform = create_transform(
            input_size=args.input_size,
            is_training=True,
            color_jitter=args.color_jitter,
            auto_augment=args.aa,
            interpolation=args.train_interpolation,
            re_prob=args.reprob,
            re_mode=args.remode,
            re_count=args.recount,
        )
        if not resize_im:
            # replace RandomResizedCropAndInterpolation with
            # RandomCrop
            transform.transforms[0] = transforms.RandomCrop(
                args.input_size, padding=4)
        return transform

    t = []
    if resize_im:
        # warping (no cropping) when evaluated at 384 or larger
        if args.input_size >= 384:
            t.append(
                transforms.Resize((args.input_size, args.input_size),
                                  interpolation=transforms.InterpolationMode.BICUBIC),
            )
            print(f"Warping {args.input_size} size input images...")
        else:
            args.crop_pct = 224 / 256
            size = int(args.input_size / args.crop_pct)
            t.append(
                transforms.Resize(size, interpolation=3),  # to maintain same ratio w.r.t. 224 images
            )
            t.append(transforms.CenterCrop(args.input_size))

    t.append(transforms.ToTensor())
    t.append(transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD))
    return transforms.Compose(t)


Overwriting /kaggle/working/TransNeXt/classification/datasets.py


### 🏋️ Pretrained Weights: TransNeXt-Base 384
 
The checkpoint is downloaded from HuggingFace: `transnext_base_384_1k_ft_1k.pth`
 
This is a **TransNeXt-Base** model fine-tuned from ImageNet-21K pretraining to ImageNet-1K with **384×384px** input resolution. The model has approximately **~89M parameters** and achieves ~86.2% top-1 accuracy on ImageNet-1K.
 
> The transfer learning strategy: keep the entire backbone's weights, replace the classification head (1000 classes → 7 ISIC classes), then fine-tune end-to-end with a very small learning rate.

In [ ]:
%cd /kaggle/working/TransNeXt/classification/weights
!wget https://huggingface.co/DaiShiResearch/transnext-base-384-1k-ft-1k/resolve/main/transnext_base_384_1k_ft_1k.pth

In [ ]:
!ls /kaggle/working/TransNeXt/classification/weights

### ⚙️ Fine-tuning Configuration
 
```python
cfg = dict(
    model='transnext_base',
    pretrain_size=224,    # original backbone pretraining resolution
    input_size=384,       # input image size during training
    drop_path=0.2,        # reduced from 0.8 (ImageNet) — less regularization for small dataset
    lr=1e-5,              # very small LR to preserve pretrained representations
    clip_grad=1.0,        # gradient clipping for stable training
    epochs=30,
    warmup_epochs=5,      # linear LR warm-up to avoid early training shock
    mixup=0.0,            # mixup disabled (not suitable for medical images)
    cutmix=0,             # cutmix disabled
    weight_decay=0.05,
)
```
 
**Rationale behind each choice:**
- `lr=1e-5` — fine-tuning requires a small LR to avoid overwriting pretrained knowledge
- `drop_path=0.2` (down from 0.8) — the dataset is much smaller than ImageNet; lighter regularization generalizes better
- `mixup=0, cutmix=0` — blending two skin lesion images creates ambiguous labels, which is harmful in a medical context
- `warmup_epochs=5` — gradually ramping up the LR in the first 5 epochs stabilizes early training
 
---

In [31]:
%%writefile /kaggle/working/TransNeXt/classification/main.py
# Copyright (c) 2015-present, Facebook, Inc.
# All rights reserved.
'''
TransNeXt
Paper: https://arxiv.org/abs/2311.17132

The training code is mainly modified based on the PVT project.
Modifications have been made to add some new features or optimize the user experience, including:
* Support for gradient accumulation under fp16 automatic mixed precision training
* Support for setting no weight decay based on parameter name keywords
* Support for accelerating training using the torch.compile mode, enhancing compatibility with the new version of PyTorch.
* Improve user usability, such as a more friendly checkpoint saving strategy, etc.

TransNeXt and PVTv2 use the **exact same** DeiT training strategy.
The above modifications do not involve changes to the model's training method. For details, see the settings in Appendix C.2 of the paper.

Author: Dai Shi
Github: https://github.com/DaiShiResearch
Email: daishiresearch@gmail.com

This source code is licensed under the license found in the
LICENSE file in the root directory of this source tree.
'''

import argparse
import datetime
import numpy as np
import time
import torch
import torch.backends.cudnn as cudnn
import json

from pathlib import Path

from timm.data import Mixup
from timm.models import create_model
from timm.loss import LabelSmoothingCrossEntropy, SoftTargetCrossEntropy
from timm.scheduler import create_scheduler

from optimizer import build_optimizer
from timm.utils import get_state_dict, ModelEma

from datasets import build_dataset
from engine import train_one_epoch, evaluate, NativeScalerAccum
from losses import DistillationLoss
from samplers import RASampler
import utils
import pkg_resources

# The following code is adapted for PyTorch version 2.1.0 and above.
# This method may not work for future versions.
torch_version = pkg_resources.get_distribution("torch").version
if torch_version >= '2.1.0':
    import torch._dynamo

    torch._dynamo.config.automatic_dynamic_shapes = False

import transnext


def get_args_parser():
    parser = argparse.ArgumentParser('TransNeXt training and evaluation script', add_help=False)
    parser.add_argument('--fp32-resume', action='store_true', default=False)
    parser.add_argument('--batch-size', default=128, type=int)
    parser.add_argument('--epochs', default=300, type=int)
    parser.add_argument('--eval-split', default='val', choices=['val', 'test'],
                    help='which split to use when is_train=False')
    parser.add_argument('--update-freq', default=1, type=int,
                        help='Number of steps to accumulate gradients when updating parameters, set to 1 to disable this feature')
    parser.add_argument('--config', required=True, type=str, help='config')

    # Model parameters
    parser.add_argument('--model', default='transnext_base', type=str, metavar='MODEL',
                        help='Name of model to train')
    parser.add_argument('--input-size', default=224, type=int, help='images input size')
    parser.add_argument('--pretrain-size', default=None, type=int, help='model pretrain size')
    parser.add_argument('--fixed-pool-size', default=None, type=int,
                        help='When this parameter is set, the model will enable linear inference mode (see section 4.1 in the paper)')

    parser.add_argument('--drop', type=float, default=0.0, metavar='PCT',
                        help='Dropout rate (default: 0.)')
    parser.add_argument('--drop-path', type=float, default=0.1, metavar='PCT',
                        help='Drop path rate (default: 0.1)')

    # parser.add_argument('--model-ema', action='store_true')
    # parser.add_argument('--no-model-ema', action='store_false', dest='model_ema')
    # parser.set_defaults(model_ema=True)
    # parser.add_argument('--model-ema-decay', type=float, default=0.99996, help='')
    # parser.add_argument('--model-ema-force-cpu', action='store_true', default=False, help='')

    # Optimizer parameters
    parser.add_argument('--opt', default='adamw', type=str, metavar='OPTIMIZER',
                        help='Optimizer (default: "adamw"')
    parser.add_argument('--opt-eps', default=1e-8, type=float, metavar='EPSILON',
                        help='Optimizer Epsilon (default: 1e-8)')
    parser.add_argument('--opt-betas', default=None, type=float, nargs='+', metavar='BETA',
                        help='Optimizer Betas (default: None, use opt default)')
    parser.add_argument('--clip-grad', type=float, default=None, metavar='NORM',
                        help='Clip gradient norm (default: None, no clipping)')
    parser.add_argument('--momentum', type=float, default=0.9, metavar='M',
                        help='SGD momentum (default: 0.9)')
    parser.add_argument('--weight-decay', type=float, default=0.05,
                        help='weight decay (default: 0.05)')
    # Learning rate schedule parameters
    parser.add_argument('--sched', default='cosine', type=str, metavar='SCHEDULER',
                        help='LR scheduler (default: "cosine"')
    parser.add_argument('--lr', type=float, default=1e-3, metavar='LR',
                        help='learning rate (default: 5e-4)')
    parser.add_argument('--lr-noise', type=float, nargs='+', default=None, metavar='pct, pct',
                        help='learning rate noise on/off epoch percentages')
    parser.add_argument('--lr-noise-pct', type=float, default=0.67, metavar='PERCENT',
                        help='learning rate noise limit percent (default: 0.67)')
    parser.add_argument('--lr-noise-std', type=float, default=1.0, metavar='STDDEV',
                        help='learning rate noise std-dev (default: 1.0)')
    parser.add_argument('--warmup-lr', type=float, default=1e-6, metavar='LR',
                        help='warmup learning rate (default: 1e-6)')
    parser.add_argument('--min-lr', type=float, default=1e-5, metavar='LR',
                        help='lower lr bound for cyclic schedulers that hit 0 (1e-5)')

    parser.add_argument('--decay-epochs', type=float, default=30, metavar='N',
                        help='epoch interval to decay LR')
    parser.add_argument('--warmup-epochs', type=int, default=5, metavar='N',
                        help='epochs to warmup LR, if scheduler supports')
    parser.add_argument('--cooldown-epochs', type=int, default=10, metavar='N',
                        help='epochs to cooldown LR at min_lr, after cyclic schedule ends')
    parser.add_argument('--patience-epochs', type=int, default=10, metavar='N',
                        help='patience epochs for Plateau LR scheduler (default: 10')
    parser.add_argument('--decay-rate', '--dr', type=float, default=0.1, metavar='RATE',
                        help='LR decay rate (default: 0.1)')

    # Augmentation parameters
    parser.add_argument('--color-jitter', type=float, default=0.4, metavar='PCT',
                        help='Color jitter factor (default: 0.4)')
    parser.add_argument('--aa', type=str, default='rand-m9-mstd0.5-inc1', metavar='NAME',
                        help='Use AutoAugment policy. "v0" or "original". " + \
                             "(default: rand-m9-mstd0.5-inc1)'),
    parser.add_argument('--smoothing', type=float, default=0.1, help='Label smoothing (default: 0.1)')
    parser.add_argument('--train-interpolation', type=str, default='bicubic',
                        help='Training interpolation (random, bilinear, bicubic default: "bicubic")')

    parser.add_argument('--repeated-aug', action='store_true')
    parser.add_argument('--no-repeated-aug', action='store_false', dest='repeated_aug')
    parser.set_defaults(repeated_aug=True)

    # * Random Erase params
    parser.add_argument('--reprob', type=float, default=0.25, metavar='PCT',
                        help='Random erase prob (default: 0.25)')
    parser.add_argument('--remode', type=str, default='pixel',
                        help='Random erase mode (default: "pixel")')
    parser.add_argument('--recount', type=int, default=1,
                        help='Random erase count (default: 1)')
    parser.add_argument('--resplit', action='store_true', default=False,
                        help='Do not random erase first (clean) augmentation split')

    # * Mixup params
    parser.add_argument('--mixup', type=float, default=0.8,
                        help='mixup alpha, mixup enabled if > 0. (default: 0.8)')
    parser.add_argument('--cutmix', type=float, default=1.0,
                        help='cutmix alpha, cutmix enabled if > 0. (default: 1.0)')
    parser.add_argument('--cutmix-minmax', type=float, nargs='+', default=None,
                        help='cutmix min/max ratio, overrides alpha and enables cutmix if set (default: None)')
    parser.add_argument('--mixup-prob', type=float, default=1.0,
                        help='Probability of performing mixup or cutmix when either/both is enabled')
    parser.add_argument('--mixup-switch-prob', type=float, default=0.5,
                        help='Probability of switching to cutmix when both mixup and cutmix enabled')
    parser.add_argument('--mixup-mode', type=str, default='batch',
                        help='How to apply mixup/cutmix params. Per "batch", "pair", or "elem"')

    # Distillation parameters
    # parser.add_argument('--teacher-model', default='regnety_160', type=str, metavar='MODEL',
    #                     help='Name of teacher model to train (default: "regnety_160"')
    # parser.add_argument('--teacher-path', type=str, default='')
    # parser.add_argument('--distillation-type', default='none', choices=['none', 'soft', 'hard'], type=str, help="")
    # parser.add_argument('--distillation-alpha', default=0.5, type=float, help="")
    # parser.add_argument('--distillation-tau', default=1.0, type=float, help="")

    # * Finetuning params
    parser.add_argument('--finetune', default='', help='finetune from checkpoint')

    # Dataset parameters
    parser.add_argument('--data-path', default='', type=str,
                        help='dataset path')
    parser.add_argument('--data-set', default='IMNET',
                    choices=['CIFAR', 'IMNET', 'INAT', 'INAT19', 'ISIC_2018'],
                    type=str, help='Image Net dataset path')
    parser.add_argument('--use-mcloader', action='store_true', default=False, help='Use mcloader')
    parser.add_argument('--inat-category', default='name',
                        choices=['kingdom', 'phylum', 'class', 'order', 'supercategory', 'family', 'genus', 'name'],
                        type=str, help='semantic granularity')

    parser.add_argument('--output_dir', default='',
                        help='path where to save, empty for no saving')
    parser.add_argument('--device', default='cuda',
                        help='device to use for training / testing')
    parser.add_argument('--seed', default=0, type=int)
    parser.add_argument('--resume', default='', help='resume from checkpoint')
    parser.add_argument('--start_epoch', default=0, type=int, metavar='N',
                        help='start epoch')
    parser.add_argument('--eval', action='store_true', help='Perform evaluation only')
    parser.add_argument('--dist-eval', action='store_true', default=False, help='Enabling distributed evaluation')
    parser.add_argument('--num_workers', default=10, type=int)
    parser.add_argument('--pin-mem', action='store_true',
                        help='Pin CPU memory in DataLoader for more efficient (sometimes) transfer to GPU.')
    parser.add_argument('--no-pin-mem', action='store_false', dest='pin_mem',
                        help='')
    parser.set_defaults(pin_mem=True)

    # Parameters for training with torch.compile mode
    parser.add_argument('--compile-model', action='store_true')
    parser.add_argument('--no-compile-model', action='store_false', dest='compile_model')
    parser.set_defaults(compile_model=True)  # Default is to use torch.compile
    parser.add_argument('--cache-size-limit', default=128, type=int)

    # distributed training parameters
    parser.add_argument('--world_size', default=1, type=int,
                        help='number of distributed processes')
    parser.add_argument('--dist_url', default='env://', help='url used to set up distributed training')
    return parser


def main(args):
    utils.init_distributed_mode(args)
    print(args)
    # if args.distillation_type != 'none' and args.finetune and not args.eval:
    #     raise NotImplementedError("Finetuning with distillation not yet supported")

    device = torch.device(args.device)

    # fix the seed for reproducibility
    seed = args.seed + utils.get_rank()
    torch.manual_seed(seed)
    np.random.seed(seed)
    # random.seed(seed)

    cudnn.benchmark = True

    dataset_train, args.nb_classes = build_dataset(is_train=True, args=args)
    dataset_val, _ = build_dataset(is_train=False, args=args)

    if True:  # args.distributed:
        num_tasks = utils.get_world_size()
        global_rank = utils.get_rank()
        if args.repeated_aug:
            sampler_train = RASampler(
                dataset_train, num_replicas=num_tasks, rank=global_rank, shuffle=True
            )
        else:
            sampler_train = torch.utils.data.DistributedSampler(
                dataset_train,
                # num_replicas=num_tasks,
                num_replicas=0,
                rank=global_rank, shuffle=True
            )
        if args.dist_eval:
            if len(dataset_val) % num_tasks != 0:
                print('Warning: Enabling distributed evaluation with an eval dataset not divisible by process number. '
                      'This will slightly alter validation results as extra duplicate entries are added to achieve '
                      'equal num of samples per-process.')
            sampler_val = torch.utils.data.DistributedSampler(
                dataset_val,
                # num_replicas=num_tasks,
                num_replicas=0,
                rank=global_rank, shuffle=False)
        else:
            sampler_val = torch.utils.data.SequentialSampler(dataset_val)
    else:
        sampler_train = torch.utils.data.RandomSampler(dataset_train)
        sampler_val = torch.utils.data.SequentialSampler(dataset_val)

    data_loader_train = torch.utils.data.DataLoader(
        dataset_train, sampler=sampler_train,
        batch_size=args.batch_size,
        num_workers=args.num_workers,
        pin_memory=args.pin_mem,
        drop_last=True,
    )

    data_loader_val = torch.utils.data.DataLoader(
        dataset_val, sampler=sampler_val,
        batch_size=int(1.5 * args.batch_size),
        num_workers=args.num_workers,
        pin_memory=args.pin_mem,
        drop_last=False
    )

    mixup_fn = None
    mixup_active = args.mixup > 0 or args.cutmix > 0. or args.cutmix_minmax is not None
    if mixup_active:
        mixup_fn = Mixup(
            mixup_alpha=args.mixup, cutmix_alpha=args.cutmix, cutmix_minmax=args.cutmix_minmax,
            prob=args.mixup_prob, switch_prob=args.mixup_switch_prob, mode=args.mixup_mode,
            label_smoothing=args.smoothing, num_classes=args.nb_classes)

    print(f"Creating model: {args.model}")
    model = create_model(
        args.model,
        img_size=args.input_size,
        pretrain_size=args.pretrain_size,
        fixed_pool_size=args.fixed_pool_size,
        pretrained=False,
        num_classes=args.nb_classes,
        drop_rate=args.drop,
        drop_path_rate=args.drop_path,
        drop_block_rate=None,
    )

    if args.finetune:
        if args.finetune.startswith('https'):
            checkpoint = torch.hub.load_state_dict_from_url(
                args.finetune, map_location='cpu', check_hash=True)
        else:
            checkpoint = torch.load(args.finetune, map_location='cpu')

        if 'model' in checkpoint:
            checkpoint_model = checkpoint['model']
        else:
            checkpoint_model = checkpoint
        state_dict = model.state_dict()
        for k in ['head.weight', 'head.bias', 'head_dist.weight', 'head_dist.bias']:
            if k in checkpoint_model and checkpoint_model[k].shape != state_dict[k].shape:
                print(f"Removing key {k} from pretrained checkpoint")
                del checkpoint_model[k]
        assert args.pretrain_size is not None, 'In finetune,you need input the pretrain size of your model'
        model.load_state_dict(checkpoint_model, strict=False)

    model.to(device)

    model_ema = None
    # if args.model_ema:
    #     # Important to create EMA model after cuda(), DP wrapper, and AMP but before SyncBN and DDP wrapper
    #     model_ema = ModelEma(
    #         model,
    #         decay=args.model_ema_decay,
    #         device='cpu' if args.model_ema_force_cpu else '',
    #         resume='')

    if args.distributed:
        model = torch.nn.parallel.DistributedDataParallel(model, device_ids=[args.gpu])
        model_without_ddp = model.module
    else:
        model_without_ddp = model
    n_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print('number of params:', n_parameters)

    # linear_scaled_lr = args.lr * args.batch_size * utils.get_world_size() / 512.0
    # args.lr = linear_scaled_lr
    total_batch_size = args.batch_size * args.update_freq * utils.get_world_size()
    print("LR = %.8f" % args.lr)
    print("Batch size = %d" % total_batch_size)
    print("Update frequent = %d" % args.update_freq)
    optimizer = build_optimizer(args, model_without_ddp)
    loss_scaler = NativeScalerAccum()
    lr_scheduler, _ = create_scheduler(args, optimizer)

    if args.mixup > 0.:
        # smoothing is handled with mixup label transform
        criterion = SoftTargetCrossEntropy()
    elif args.smoothing:
        criterion = LabelSmoothingCrossEntropy(smoothing=args.smoothing)
    else:
        criterion = torch.nn.CrossEntropyLoss()

    # teacher_model = None
    # if args.distillation_type != 'none':
    #     assert args.teacher_path, 'need to specify teacher-path when using distillation'
    #     print(f"Creating teacher model: {args.teacher_model}")
    #     teacher_model = create_model(
    #         args.teacher_model,
    #         pretrained=False,
    #         num_classes=args.nb_classes,
    #         global_pool='avg',
    #     )
    #     if args.teacher_path.startswith('https'):
    #         checkpoint = torch.hub.load_state_dict_from_url(
    #             args.teacher_path, map_location='cpu', check_hash=True)
    #     else:
    #         checkpoint = torch.load(args.teacher_path, map_location='cpu')
    #     teacher_model.load_state_dict(checkpoint['model'])
    #     teacher_model.to(device)
    #     teacher_model.eval()

    # wrap the criterion in our custom DistillationLoss, which
    # just dispatches to the original criterion if args.distillation_type is 'none'
    # criterion = DistillationLoss(
    #     criterion, teacher_model, args.distillation_type, args.distillation_alpha, args.distillation_tau
    # )
    criterion = DistillationLoss(
        criterion, None, 'none', 0, 0
    )

    output_dir = Path(args.output_dir)

    max_accuracy = 0.0

    if args.resume:
        if args.resume.startswith('https'):
            checkpoint = torch.hub.load_state_dict_from_url(
                args.resume, map_location='cpu', check_hash=True)
        else:
            checkpoint = torch.load(args.resume, map_location='cpu')

        checkpoint_model = checkpoint['model'] if 'model' in checkpoint else checkpoint
        msg = model_without_ddp.load_state_dict(checkpoint_model)

        print(msg)
        if not args.eval and 'optimizer' in checkpoint and 'lr_scheduler' in checkpoint and 'epoch' in checkpoint:
            print('Loading optimizer checkpoint')
            optimizer.load_state_dict(checkpoint['optimizer'])
            if lr_scheduler is not None:
                print('Loading lr_scheduler checkpoint')
                lr_scheduler.load_state_dict(checkpoint['lr_scheduler'])
            args.start_epoch = checkpoint['epoch'] + 1
            # if args.model_ema:
            #     utils._load_checkpoint_for_ema(model_ema, checkpoint['model_ema'])
            if 'scaler' in checkpoint:
                print('Loading scaler checkpoint')
                loss_scaler.load_state_dict(checkpoint['scaler'])
            if 'max_accuracy' in checkpoint:
                max_accuracy = checkpoint['max_accuracy']
                print(f'Previous max accuracy record is {max_accuracy:.2f}%')

    if args.compile_model:
        model = torch.compile(model)
        torch._dynamo.config.cache_size_limit = args.cache_size_limit

    if args.eval:
        test_stats = evaluate(data_loader_val, model, device)
        print(f"Accuracy of the network on the {len(dataset_val)} test images: {test_stats['acc1']:.1f}%")
        return

    print(f"Start training for {args.epochs} epochs")
    start_time = time.time()

    for epoch in range(args.start_epoch, args.epochs):
        if args.fp32_resume and epoch > args.start_epoch + 1:
            args.fp32_resume = False
        loss_scaler._scaler = torch.cuda.amp.GradScaler(enabled=not args.fp32_resume)

        if args.distributed:
            data_loader_train.sampler.set_epoch(epoch)

        train_stats = train_one_epoch(
            model, criterion, data_loader_train,
            optimizer, device, epoch, loss_scaler,
            args.clip_grad, model_ema, mixup_fn,
            set_training_mode=args.finetune == '',  # keep in eval mode during finetuning
            fp32=args.fp32_resume,
            grad_accum_steps=args.update_freq,
        )

        if lr_scheduler is not None:
            lr_scheduler.step(epoch)
        if args.output_dir:
            checkpoint_paths = [output_dir / 'last.pth']
            checkpoint_dict = {'model': model_without_ddp.state_dict(),
                               'optimizer': optimizer.state_dict(),
                               'lr_scheduler': lr_scheduler.state_dict() if lr_scheduler is not None else None,
                               'epoch': epoch,
                               # 'model_ema': get_state_dict(model_ema),
                               'scaler': loss_scaler.state_dict(),
                               'args': args,
                               'max_accuracy': max_accuracy}

            for checkpoint_path in checkpoint_paths:
                utils.save_on_master(checkpoint_dict, checkpoint_path)

        test_stats = evaluate(data_loader_val, model, device)
        print(f"Accuracy of the network on the {len(dataset_val)} test images: {test_stats['acc1']:.1f}%")
        if test_stats['acc1'] >= max_accuracy and args.output_dir:
            checkpoint_paths = [output_dir / 'best.pth']
            for checkpoint_path in checkpoint_paths:
                utils.save_on_master(checkpoint_dict, checkpoint_path)
                print(f'Saving best performance checkpoint')
        max_accuracy = max(max_accuracy, test_stats["acc1"])
        print(f'Max accuracy: {max_accuracy:.2f}%')

        log_stats = {**{f'train_{k}': v for k, v in train_stats.items()},
                     **{f'test_{k}': v for k, v in test_stats.items()},
                     'epoch': epoch,
                     'n_parameters': n_parameters}

        if args.output_dir and utils.is_main_process():
            with (output_dir / "log.txt").open("a") as f:
                f.write(json.dumps(log_stats) + "\n")

    total_time = time.time() - start_time
    total_time_str = str(datetime.timedelta(seconds=int(total_time)))
    print('Training time {}'.format(total_time_str))


if __name__ == '__main__':
    parser = argparse.ArgumentParser('DeiT training and evaluation script', parents=[get_args_parser()])
    args = parser.parse_args()
    args = utils.update_from_config(args)
    if args.output_dir:
        Path(args.output_dir).mkdir(parents=True, exist_ok=True)
    main(args)


Overwriting /kaggle/working/TransNeXt/classification/main.py


In [16]:
%%writefile /kaggle/working/TransNeXt/classification/transnext.py
'''
TransNeXt: Robust Foveal Visual Perception for Vision Transformers
Paper: https://arxiv.org/abs/2311.17132
Code: https://github.com/DaiShiResearch/TransNeXt

Author: Dai Shi
Github: https://github.com/DaiShiResearch
Email: daishiresearch@gmail.com

This source code is licensed under the license found in the
LICENSE file in the root directory of this source tree.
'''

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from functools import partial
from timm.models.layers import DropPath, to_2tuple, trunc_normal_
from timm.models.registry import register_model
from timm.models.vision_transformer import _cfg
import math
import pkg_resources


def is_installed(package_name):
    try:
        pkg_resources.get_distribution(package_name)
        return True
    except pkg_resources.DistributionNotFound:
        return False


if is_installed('swattention'):
    print('swattention package found, loading CUDA version of Aggregated Attention')
    from attention_cuda import AggregatedAttention
else:
    print('swattention package not found, loading PyTorch native version of Aggregated Attention')
    from attention_native import AggregatedAttention


class DWConv(nn.Module):
    def __init__(self, dim=768):
        super(DWConv, self).__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=3, stride=1, padding=1, bias=True, groups=dim)

    def forward(self, x, H, W):
        B, N, C = x.shape
        x = x.transpose(1, 2).view(B, C, H, W).contiguous()
        x = self.dwconv(x)
        x = x.flatten(2).transpose(1, 2)

        return x


class ConvolutionalGLU(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        hidden_features = int(2 * hidden_features / 3)
        self.fc1 = nn.Linear(in_features, hidden_features * 2)
        self.dwconv = DWConv(hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x, H, W):
        x, v = self.fc1(x).chunk(2, dim=-1)
        x = self.act(self.dwconv(x, H, W)) * v
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


@torch.no_grad()
def get_relative_position_cpb(query_size, key_size, pretrain_size=None):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    pretrain_size = pretrain_size or query_size
    axis_qh = torch.arange(query_size[0], dtype=torch.float32, device=device)
    axis_kh = F.adaptive_avg_pool1d(axis_qh.unsqueeze(0), key_size[0]).squeeze(0)
    axis_qw = torch.arange(query_size[1], dtype=torch.float32, device=device)
    axis_kw = F.adaptive_avg_pool1d(axis_qw.unsqueeze(0), key_size[1]).squeeze(0)
    axis_kh, axis_kw = torch.meshgrid(axis_kh, axis_kw)
    axis_qh, axis_qw = torch.meshgrid(axis_qh, axis_qw)

    axis_kh = torch.reshape(axis_kh, [-1])
    axis_kw = torch.reshape(axis_kw, [-1])
    axis_qh = torch.reshape(axis_qh, [-1])
    axis_qw = torch.reshape(axis_qw, [-1])

    relative_h = (axis_qh[:, None] - axis_kh[None, :]) / (pretrain_size[0] - 1) * 8
    relative_w = (axis_qw[:, None] - axis_kw[None, :]) / (pretrain_size[1] - 1) * 8
    relative_hw = torch.stack([relative_h, relative_w], dim=-1).view(-1, 2)

    relative_coords_table, idx_map = torch.unique(relative_hw, return_inverse=True, dim=0)

    relative_coords_table = torch.sign(relative_coords_table) * torch.log2(
        torch.abs(relative_coords_table) + 1.0) / torch.log2(torch.tensor(8, dtype=torch.float32))

    return idx_map, relative_coords_table


class Attention(nn.Module):
    def __init__(self, dim, input_resolution, num_heads=8, qkv_bias=True, attn_drop=0.,
                 proj_drop=0.):
        super().__init__()
        assert dim % num_heads == 0, f"dim {dim} should be divided by num_heads {num_heads}."

        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.temperature = nn.Parameter(
            torch.log((torch.ones(num_heads, 1, 1) / 0.24).exp() - 1))  # Initialize softplus(temperature) to 1/0.24.
        # Generate sequnce length scale
        self.register_buffer("seq_length_scale", torch.as_tensor(np.log(input_resolution[0] * input_resolution[1])),
                             persistent=False)

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.query_embedding = nn.Parameter(
            nn.init.trunc_normal_(torch.empty(self.num_heads, 1, self.head_dim), mean=0, std=0.02))

        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

        # mlp to generate continuous relative position bias
        self.cpb_fc1 = nn.Linear(2, 512, bias=True)
        self.cpb_act = nn.ReLU(inplace=True)
        self.cpb_fc2 = nn.Linear(512, num_heads, bias=True)

    def forward(self, x, H, W, relative_pos_index, relative_coords_table):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, -1, 3 * self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        q, k, v = qkv.chunk(3, dim=1)

        # Use MLP to generate continuous relative positional bias
        rel_bias = self.cpb_fc2(self.cpb_act(self.cpb_fc1(relative_coords_table))).transpose(0, 1)[:,
                   relative_pos_index.view(-1)].view(-1, N, N)

        # Calculate attention map using sequence length scaled cosine attention and query embedding
        attn = ((F.normalize(q, dim=-1) + self.query_embedding) * F.softplus(
            self.temperature) * self.seq_length_scale) @ F.normalize(k, dim=-1).transpose(-2, -1) + rel_bias
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


class Block(nn.Module):

    def __init__(self, dim, num_heads, input_resolution, window_size=3, mlp_ratio=4.,
                 qkv_bias=False, drop=0., attn_drop=0.,
                 drop_path=0., act_layer=nn.GELU, norm_layer=nn.LayerNorm, sr_ratio=1, fixed_pool_size=None):
        super().__init__()
        self.norm1 = norm_layer(dim)
        if sr_ratio == 1:
            self.attn = Attention(
                dim,
                input_resolution,
                num_heads=num_heads,
                qkv_bias=qkv_bias,
                attn_drop=attn_drop,
                proj_drop=drop)
        else:
            self.attn = AggregatedAttention(
                dim,
                input_resolution,
                window_size=window_size,
                num_heads=num_heads,
                qkv_bias=qkv_bias,
                attn_drop=attn_drop,
                proj_drop=drop,
                sr_ratio=sr_ratio,
                fixed_pool_size=fixed_pool_size)
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = ConvolutionalGLU(in_features=dim, hidden_features=mlp_hidden_dim, act_layer=act_layer, drop=drop)

        # NOTE: drop path for stochastic depth, we shall see if this is better than dropout here
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()

    def forward(self, x, H, W, relative_pos_index, relative_coords_table):
        x = x + self.drop_path(self.attn(self.norm1(x), H, W, relative_pos_index, relative_coords_table))
        x = x + self.drop_path(self.mlp(self.norm2(x), H, W))

        return x


class OverlapPatchEmbed(nn.Module):
    """ Image to Patch Embedding
    """

    def __init__(self, patch_size=7, stride=4, in_chans=3, embed_dim=768):
        super().__init__()

        patch_size = to_2tuple(patch_size)

        assert max(patch_size) > stride, "Set larger patch_size than stride"
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=stride,
                              padding=(patch_size[0] // 2, patch_size[1] // 2))
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.proj(x)
        _, _, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)
        x = self.norm(x)

        return x, H, W


class TransNeXt(nn.Module):
    '''
    The parameter "img size" is primarily utilized for generating relative spatial coordinates,
    which are used to compute continuous relative positional biases. As this TransNeXt implementation does not support multi-scale inputs,
    it is recommended to set the "img size" parameter to a value that is exactly the same as the resolution of the inference images.
    It is not advisable to set the "img size" parameter to a value exceeding 800x800.
    The "pretrain size" refers to the "img size" used during the initial pre-training phase,
    which is used to scale the relative spatial coordinates for better extrapolation by the MLP.
    For models trained on ImageNet-1K at a resolution of 224x224,
    as well as downstream task models fine-tuned based on these pre-trained weights,
    the "pretrain size" parameter should be set to 224x224.
    '''

    def __init__(self, img_size=224, pretrain_size=None, window_size=[3, 3, 3, None],
                 patch_size=16, in_chans=3, num_classes=1000, embed_dims=[64, 128, 256, 512],
                 num_heads=[1, 2, 4, 8], mlp_ratios=[4, 4, 4, 4], qkv_bias=False, drop_rate=0.,
                 attn_drop_rate=0., drop_path_rate=0., norm_layer=nn.LayerNorm,
                 depths=[3, 4, 6, 3], sr_ratios=[8, 4, 2, 1], num_stages=4, fixed_pool_size=None):
        super().__init__()
        self.num_classes = num_classes
        self.depths = depths
        self.num_stages = num_stages
        pretrain_size = pretrain_size or img_size

        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]  # stochastic depth decay rule
        cur = 0

        for i in range(num_stages):
            # Generate relative positional coordinate table and index for each stage to compute continuous relative positional bias.
            relative_pos_index, relative_coords_table = get_relative_position_cpb(
                query_size=to_2tuple(img_size // (2 ** (i + 2))),
                key_size=to_2tuple(img_size // ((2 ** (i + 2)) * sr_ratios[i])) if (
                        fixed_pool_size is None or sr_ratios[i] == 1) else to_2tuple(fixed_pool_size),
                pretrain_size=to_2tuple(pretrain_size // (2 ** (i + 2))))

            self.register_buffer(f"relative_pos_index{i + 1}", relative_pos_index, persistent=False)
            self.register_buffer(f"relative_coords_table{i + 1}", relative_coords_table, persistent=False)

            patch_embed = OverlapPatchEmbed(patch_size=patch_size * 2 - 1 if i == 0 else 3,
                                            stride=patch_size if i == 0 else 2,
                                            in_chans=in_chans if i == 0 else embed_dims[i - 1],
                                            embed_dim=embed_dims[i])

            block = nn.ModuleList([Block(
                dim=embed_dims[i], input_resolution=to_2tuple(img_size // (2 ** (i + 2))), window_size=window_size[i],
                num_heads=num_heads[i], mlp_ratio=mlp_ratios[i], qkv_bias=qkv_bias,
                drop=drop_rate, attn_drop=attn_drop_rate, drop_path=dpr[cur + j], norm_layer=norm_layer,
                sr_ratio=sr_ratios[i], fixed_pool_size=fixed_pool_size)
                for j in range(depths[i])])
            norm = norm_layer(embed_dims[i])
            cur += depths[i]

            setattr(self, f"patch_embed{i + 1}", patch_embed)
            setattr(self, f"block{i + 1}", block)
            setattr(self, f"norm{i + 1}", norm)

        # classification head
        self.head = nn.Linear(embed_dims[3], num_classes) if num_classes > 0 else nn.Identity()

        for n, m in self.named_modules():
            self._init_weights(m, n)

    def _init_weights(self, m: nn.Module, name: str = ''):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Conv2d):
            fan_out = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()
        elif isinstance(m, (nn.LayerNorm, nn.GroupNorm, nn.BatchNorm2d)):
            nn.init.zeros_(m.bias)
            nn.init.ones_(m.weight)

    @torch.jit.ignore
    def no_weight_decay(self):
        return {}

    @torch.jit.ignore
    def no_weight_decay_keywords(self):
        return {'query_embedding', 'relative_pos_bias_local', 'cpb', 'temperature'}

    def get_classifier(self):
        return self.head

    def reset_classifier(self, num_classes, global_pool=''):
        self.num_classes = num_classes
        self.head = nn.Linear(self.embed_dim, num_classes) if num_classes > 0 else nn.Identity()

    def forward_features(self, x):
        B = x.shape[0]

        for i in range(self.num_stages):
            patch_embed = getattr(self, f"patch_embed{i + 1}")
            block = getattr(self, f"block{i + 1}")
            norm = getattr(self, f"norm{i + 1}")
            x, H, W = patch_embed(x)
            relative_pos_index = getattr(self, f"relative_pos_index{i + 1}")
            relative_coords_table = getattr(self, f"relative_coords_table{i + 1}")
            for blk in block:
                x = blk(x, H, W, relative_pos_index, relative_coords_table)
            x = norm(x)
            if i != self.num_stages - 1:
                x = x.reshape(B, H, W, -1).permute(0, 3, 1, 2).contiguous()

        return x.mean(dim=1)

    def forward(self, x):
        x = self.forward_features(x)
        x = self.head(x)

        return x


@register_model
def transnext_micro(pretrained=False, **kwargs):
    model = TransNeXt(window_size=[3, 3, 3, None],
                      patch_size=4, embed_dims=[48, 96, 192, 384], num_heads=[2, 4, 8, 16],
                      mlp_ratios=[8, 8, 4, 4], qkv_bias=True,
                      norm_layer=partial(nn.LayerNorm, eps=1e-6), depths=[2, 2, 15, 2], sr_ratios=[8, 4, 2, 1],
                      **kwargs)
    model.default_cfg = _cfg()

    return model


@register_model
def transnext_tiny(pretrained=False, **kwargs):
    model = TransNeXt(window_size=[3, 3, 3, None],
                      patch_size=4, embed_dims=[72, 144, 288, 576], num_heads=[3, 6, 12, 24],
                      mlp_ratios=[8, 8, 4, 4], qkv_bias=True,
                      norm_layer=partial(nn.LayerNorm, eps=1e-6), depths=[2, 2, 15, 2], sr_ratios=[8, 4, 2, 1],
                      **kwargs)
    model.default_cfg = _cfg()

    return model


@register_model
def transnext_small(pretrained=False, **kwargs):
    model = TransNeXt(window_size=[3, 3, 3, None],
                      patch_size=4, embed_dims=[72, 144, 288, 576], num_heads=[3, 6, 12, 24],
                      mlp_ratios=[8, 8, 4, 4], qkv_bias=True,
                      norm_layer=partial(nn.LayerNorm, eps=1e-6), depths=[5, 5, 22, 5], sr_ratios=[8, 4, 2, 1],
                      **kwargs)
    model.default_cfg = _cfg()

    return model


@register_model
def transnext_base(pretrained=False, **kwargs):
    model = TransNeXt(window_size=[3, 3, 3, None],
                      patch_size=4, embed_dims=[96, 192, 384, 768], num_heads=[4, 8, 16, 32],
                      mlp_ratios=[8, 8, 4, 4], qkv_bias=True,
                      norm_layer=partial(nn.LayerNorm, eps=1e-6), depths=[5, 5, 23, 5], sr_ratios=[8, 4, 2, 1],
                      **kwargs)
    model.default_cfg = _cfg()

    return model


@register_model
def transnext_micro_AAAA(pretrained=False, **kwargs):
    model = TransNeXt(window_size=[3, 3, 3, 3],
                      patch_size=4, embed_dims=[48, 96, 192, 384], num_heads=[2, 4, 8, 16],
                      mlp_ratios=[8, 8, 4, 4], qkv_bias=True,
                      norm_layer=partial(nn.LayerNorm, eps=1e-6), depths=[2, 2, 15, 2], sr_ratios=[16, 8, 4, 2],
                      **kwargs)
    model.default_cfg = _cfg()

    return model


Overwriting /kaggle/working/TransNeXt/classification/transnext.py


In [17]:
%%writefile /kaggle/working/TransNeXt/classification/configs/finetune/transnext_base_384_ft.py
cfg = dict(
    model='transnext_base',
    pretrain_size=224,
    input_size=384,
    drop_path=0.2,       # ← giảm từ 0.8
    lr=1e-5,
    clip_grad=1.0,
    epochs=30,
    warmup_epochs=5,    # ← thêm mới
    mixup=0.0,           # ← thêm mới
    cutmix=0,
    sched=None,
    weight_decay=0.05,
    output_dir='checkpoints/transnext_base_384',
)

Overwriting /kaggle/working/TransNeXt/classification/configs/finetune/transnext_base_384_ft.py


### 🎯 Training Results (30 Epochs)
 
Training ran across **2 GPUs** in parallel (`dist_train.sh ... 2`), with batch size 8 per GPU (16 effective images per step).
 
**Final results on the Val set (193 images):**
 
| Metric | Value |
|---|---|
| **Val Acc@1** | **93.78%** |
| Val Acc@5 | 98.96% |
| Final training loss | ~0.575 |
| Total training time | **6 hours 31 minutes** |
 
> The best checkpoint is automatically saved to `checkpoints/transnext_base_384/best.pth`. The final epoch (29) shows training loss ≈ 0.575, indicating the model has converged well.
 
> **Note:** The val set only has 193 images, so the val accuracy carries significant variance. A rigorous evaluation requires the **test set (1,512 images)** — covered in the next section.


In [17]:
%cd /kaggle/working/TransNeXt/classification
!bash dist_train.sh ./configs/finetune/transnext_base_384_ft.py \
    2 \
    --data-path /kaggle/tmp/isic_2018_task3 \
    --batch-size 8 \
    --finetune  /kaggle/working/TransNeXt/classification/weights/transnext_base_384_1k_ft_1k.pth \
    --eval-split val \
    --data-set ISIC_2018

/kaggle/working/TransNeXt/classification
/kaggle/working/venv/lib/python3.8/site-packages/torch/distributed/launch.py:181: FutureWarning: The module torch.distributed.launch is deprecated
and will be removed in future. Use torchrun.
Note that --use-env is set by default in torchrun.
If your script expects `--local-rank` argument to be set, please
change it to read from `os.environ['LOCAL_RANK']` instead. See 
https://pytorch.org/docs/stable/distributed.html#launch-utility for 
further instructions

  warnings.warn(
*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
/kaggle/working/venv/lib/python3.8/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the traini

In [20]:
!zip -r /kaggle/working/TransNeXt/classification/checkpoints.zip /kaggle/working/TransNeXt/classification/checkpoints

  adding: kaggle/working/TransNeXt/classification/checkpoints/ (stored 0%)
  adding: kaggle/working/TransNeXt/classification/checkpoints/transnext_base_384/ (stored 0%)
  adding: kaggle/working/TransNeXt/classification/checkpoints/transnext_base_384/best.pth (deflated 8%)
  adding: kaggle/working/TransNeXt/classification/checkpoints/transnext_base_384/log.txt (deflated 80%)
  adding: kaggle/working/TransNeXt/classification/checkpoints/transnext_base_384/last.pth (deflated 8%)


### 📋 Result 1: Input 384×384 — Baseline
`--eval-split test --batch-size 8`
 
| Acc@1 | Acc@5 | Loss |
|---|---|---|
| **86.8%** | 99.3% | 0.518 |
 
This is the baseline — evaluating at the same 384px resolution the model was trained on. The drop from val (93.8%) to test (86.8%) is largely explained by the very small val set (193 images) giving an overly optimistic estimate.
 
---
 
### 📋 Result 2: Input 512×512 — No pretrain-size hint
`--input-size 512 --batch-size 32`
 
| Acc@1 | Acc@5 | Loss |
|---|---|---|
| **87.7%** | 99.1% | 0.498 |
 
Scaling up to 512px improves by +0.9% over the baseline. However, without `--pretrain-size`, the model does not know how to correctly scale the attention window's positional bias relative to its training resolution.
 
---
 
### 📋 Result 3: Input 640×640 — No pretrain-size hint
`--input-size 640 --batch-size 16`
 
| Acc@1 | Acc@5 | Loss |
|---|---|---|
| **86.0%** | 99.3% | 0.507 |
 
Scaling to 640px actually drops below the 384px baseline. Without a `--pretrain-size` hint, the positional bias becomes miscalibrated when the input resolution is too far from the training resolution.
 
---
 
### 📋 Result 4: Input 512×512 + fixed_pool_size=7
`--input-size 512 --fixed-pool-size 7 --batch-size 32`
 
| Acc@1 | Acc@5 | Loss |
|---|---|---|
| **86.3%** | 99.3% | 0.515 |
 
`fixed_pool_size=7` locks the pooling window size — an alternative way to control attention scale when changing resolution. Result (86.3%) is better than 640px but worse than plain 512px, suggesting this pooling constraint limits the model's ability to leverage the higher resolution.
 
---
 
### 📋 Result 5: Input 512×512 + pretrain-size=384 ✅ BEST
`--input-size 512 --pretrain-size 384 --batch-size 16`
 
| Acc@1 | Acc@5 | Loss |
|---|---|---|
| 🏆 **88.3%** | 99.3% | 0.476 |
 
**Best result overall.** Providing `--pretrain-size 384` tells the model that its attention window was calibrated at 384px, allowing it to correctly interpolate positional biases when running at 512px. This yields a **+1.5% gain** over baseline and **+0.6%** over 512px without the hint — with no retraining required.
 
---
 
### 📋 Result 6: Input 640×640 + pretrain-size=512
`--input-size 640 --pretrain-size 512 --batch-size 16`
 
| Acc@1 | Acc@5 | Loss |
|---|---|---|
| **87.2%** | 99.3% | 0.494 |
 
With a pretrain-size hint of 512 at inference resolution 640, results improve over the no-hint 640px case (+1.2%), but still fall short of the 512px+hint configuration.
 
---
 
## 📈 Final Summary
 
| Configuration | Input Size | pretrain_size | Test Acc@1 |
|---|---|---|---|
| Baseline (train resolution) | 384 | — | 86.8% |
| Scale up | 512 | — | 87.7% |
| Scale up | 640 | — | 86.0% |
| Fixed pool size | 512 + pool=7 | — | 86.3% |
| **✅ Best** | **512** | **384** | **88.3%** |
| Larger scale | 640 | 512 | 87.2% |
 
**Key takeaways:**
- TransNeXt-Base fine-tuned on ISIC 2018 Task 3 achieves **88.3% top-1 accuracy** on the 1,512-image test set.
- **Test-time resolution scaling with a `pretrain_size` hint** is the most effective post-training improvement, gaining +1.5% over baseline with zero additional training cost.
- Scaling without the hint at high resolutions (640px) can hurt accuracy — the positional bias miscalibration outweighs the benefit of higher resolution.
- The **severe class imbalance** (NV = 60% of test set) inflates overall accuracy; per-class metrics like balanced accuracy or per-class AUC would give a more honest evaluation.
- Total training time: ~**6.5 hours** on 2× GPU (Kaggle environment).

In [17]:
%%writefile /kaggle/working/TransNeXt/classification/utils.py
# Copyright (c) 2015-present, Facebook, Inc.
# All rights reserved.
"""
Misc functions, including distributed helpers.

Mostly copy-paste from torchvision references.
"""
import io
import os
import time
from collections import defaultdict, deque
import datetime

import torch
import torch.distributed as dist
import mmcv

class SmoothedValue(object):
    """Track a series of values and provide access to smoothed values over a
    window or the global series average.
    """

    def __init__(self, window_size=20, fmt=None):
        if fmt is None:
            fmt = "{median:.4f} ({global_avg:.4f})"
        self.deque = deque(maxlen=window_size)
        self.total = 0.0
        self.count = 0
        self.fmt = fmt

    def update(self, value, n=1):
        self.deque.append(value)
        self.count += n
        self.total += value * n

    def synchronize_between_processes(self):
        """
        Warning: does not synchronize the deque!
        """
        if not is_dist_avail_and_initialized():
            return
        t = torch.tensor([self.count, self.total], dtype=torch.float64, device='cuda')
        dist.barrier()
        dist.all_reduce(t)
        t = t.tolist()
        self.count = int(t[0])
        self.total = t[1]

    @property
    def median(self):
        d = torch.tensor(list(self.deque))
        return d.median().item()

    @property
    def avg(self):
        d = torch.tensor(list(self.deque), dtype=torch.float32)
        return d.mean().item()

    @property
    def global_avg(self):
        return self.total / self.count

    @property
    def max(self):
        return max(self.deque)

    @property
    def value(self):
        return self.deque[-1]

    def __str__(self):
        return self.fmt.format(
            median=self.median,
            avg=self.avg,
            global_avg=self.global_avg,
            max=self.max,
            value=self.value)


class MetricLogger(object):
    def __init__(self, delimiter="\t"):
        self.meters = defaultdict(SmoothedValue)
        self.delimiter = delimiter

    def update(self, **kwargs):
        for k, v in kwargs.items():
            if isinstance(v, torch.Tensor):
                v = v.item()
            assert isinstance(v, (float, int))
            self.meters[k].update(v)

    def __getattr__(self, attr):
        if attr in self.meters:
            return self.meters[attr]
        if attr in self.__dict__:
            return self.__dict__[attr]
        raise AttributeError("'{}' object has no attribute '{}'".format(
            type(self).__name__, attr))

    def __str__(self):
        loss_str = []
        for name, meter in self.meters.items():
            loss_str.append(
                "{}: {}".format(name, str(meter))
            )
        return self.delimiter.join(loss_str)

    def synchronize_between_processes(self):
        for meter in self.meters.values():
            meter.synchronize_between_processes()

    def add_meter(self, name, meter):
        self.meters[name] = meter

    def log_every(self, iterable, print_freq, header=None):
        i = 0
        if not header:
            header = ''
        start_time = time.time()
        end = time.time()
        iter_time = SmoothedValue(fmt='{avg:.4f}')
        data_time = SmoothedValue(fmt='{avg:.4f}')
        space_fmt = ':' + str(len(str(len(iterable)))) + 'd'
        log_msg = [
            header,
            '[{0' + space_fmt + '}/{1}]',
            'eta: {eta}',
            '{meters}',
            'time: {time}',
            'data: {data}'
        ]
        if torch.cuda.is_available():
            log_msg.append('max mem: {memory:.0f}')
        log_msg = self.delimiter.join(log_msg)
        MB = 1024.0 * 1024.0
        for obj in iterable:
            data_time.update(time.time() - end)
            yield obj
            iter_time.update(time.time() - end)
            if i % print_freq == 0 or i == len(iterable) - 1:
                eta_seconds = iter_time.global_avg * (len(iterable) - i)
                eta_string = str(datetime.timedelta(seconds=int(eta_seconds)))
                if torch.cuda.is_available():
                    print(log_msg.format(
                        i, len(iterable), eta=eta_string,
                        meters=str(self),
                        time=str(iter_time), data=str(data_time),
                        memory=torch.cuda.max_memory_allocated() / MB))
                else:
                    print(log_msg.format(
                        i, len(iterable), eta=eta_string,
                        meters=str(self),
                        time=str(iter_time), data=str(data_time)))
            i += 1
            end = time.time()
        total_time = time.time() - start_time
        total_time_str = str(datetime.timedelta(seconds=int(total_time)))
        print('{} Total time: {} ({:.4f} s / it)'.format(
            header, total_time_str, total_time / len(iterable)))


def _load_checkpoint_for_ema(model_ema, checkpoint):
    """
    Workaround for ModelEma._load_checkpoint to accept an already-loaded object
    """
    mem_file = io.BytesIO()
    torch.save(checkpoint, mem_file)
    mem_file.seek(0)
    model_ema._load_checkpoint(mem_file)


def setup_for_distributed(is_master):
    """
    This function disables printing when not in master process
    """
    import builtins as __builtin__
    builtin_print = __builtin__.print

    def print(*args, **kwargs):
        force = kwargs.pop('force', False)
        if is_master or force:
            builtin_print(*args, **kwargs)

    __builtin__.print = print


def is_dist_avail_and_initialized():
    if not dist.is_available():
        return False
    if not dist.is_initialized():
        return False
    return True


def get_world_size():
    if not is_dist_avail_and_initialized():
        return 1
    return dist.get_world_size()


def get_rank():
    if not is_dist_avail_and_initialized():
        return 0
    return dist.get_rank()


def is_main_process():
    return get_rank() == 0


def save_on_master(*args, **kwargs):
    if is_main_process():
        torch.save(*args, **kwargs)


def init_distributed_mode(args):
    if 'RANK' in os.environ and 'WORLD_SIZE' in os.environ:
        args.rank = int(os.environ["RANK"])
        args.world_size = int(os.environ['WORLD_SIZE'])
        args.gpu = int(os.environ['LOCAL_RANK'])
    elif 'SLURM_PROCID' in os.environ:
        args.rank = int(os.environ['SLURM_PROCID'])
        args.gpu = args.rank % torch.cuda.device_count()
    else:
        print('Not using distributed mode')
        args.distributed = False
        return

    args.distributed = True

    torch.cuda.set_device(args.gpu)
    args.dist_backend = 'nccl'
    print('| distributed init (rank {}): {}'.format(
        args.rank, args.dist_url), flush=True)
    torch.distributed.init_process_group(backend=args.dist_backend, init_method=args.dist_url,
                                         world_size=args.world_size, rank=args.rank)
    torch.distributed.barrier()
    setup_for_distributed(args.rank == 0)


# def update_from_config(args):
#     cfg = mmcv.Config.fromfile(args.config)
#     for _, cfg_item in cfg._cfg_dict.items():
#         for k, v in cfg_item.items():
#             setattr(args, k, v)
#     return args

def update_from_config(args):
    cfg = mmcv.Config.fromfile(args.config)
    cli_overrides = {'input_size', 'pretrain_size', 'batch_size', 'resume', 'eval_split'}
    for _, cfg_item in cfg._cfg_dict.items():
        for k, v in cfg_item.items():
            if k not in cli_overrides:
                setattr(args, k, v)
    return args


Overwriting /kaggle/working/TransNeXt/classification/utils.py


In [18]:
!ls /kaggle/working/TransNeXt/classification/checkpoints/transnext_base_384

best.pth  last.pth  log.txt


In [35]:
%cd /kaggle/working/TransNeXt/classification
!bash dist_train.sh ./configs/finetune/transnext_base_384_ft.py \
    1 \
    --data-path /kaggle/tmp/isic_2018_task3 \
    --resume ./checkpoints/transnext_base_384/best.pth \
    --eval-split test \
    --data-set ISIC_2018 \
    --batch-size 8 \
    --eval

/kaggle/working/TransNeXt/classification
/kaggle/working/venv/lib/python3.8/site-packages/torch/distributed/launch.py:181: FutureWarning: The module torch.distributed.launch is deprecated
and will be removed in future. Use torchrun.
Note that --use-env is set by default in torchrun.
If your script expects `--local-rank` argument to be set, please
change it to read from `os.environ['LOCAL_RANK']` instead. See 
https://pytorch.org/docs/stable/distributed.html#launch-utility for 
further instructions

  warnings.warn(
/kaggle/working/venv/lib/python3.8/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(
swattention package found, loading CUDA version

In [19]:
%cd /kaggle/working/TransNeXt/classification
!bash dist_train.sh ./configs/finetune/transnext_base_384_ft.py \
  1 \
  --data-path /kaggle/tmp/isic_2018_task3 \
  --data-set ISIC_2018 \
  --resume ./checkpoints/transnext_base_384/best.pth \
  --eval \
  --eval-split test \
  --input-size 512 \
  --batch-size 32 \
  --num_workers 4

/kaggle/working/TransNeXt/classification
/kaggle/working/venv/lib/python3.8/site-packages/torch/distributed/launch.py:181: FutureWarning: The module torch.distributed.launch is deprecated
and will be removed in future. Use torchrun.
Note that --use-env is set by default in torchrun.
If your script expects `--local-rank` argument to be set, please
change it to read from `os.environ['LOCAL_RANK']` instead. See 
https://pytorch.org/docs/stable/distributed.html#launch-utility for 
further instructions

  warnings.warn(
/kaggle/working/venv/lib/python3.8/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(
swattention package found, loading CUDA version

In [22]:
%cd /kaggle/working/TransNeXt/classification
!bash dist_train.sh ./configs/finetune/transnext_base_384_ft.py \
  1 \
  --data-path /kaggle/tmp/isic_2018_task3 \
  --data-set ISIC_2018 \
  --resume ./checkpoints/transnext_base_384/best.pth \
  --eval \
  --eval-split test \
  --input-size 640 \
  --batch-size 16 \
  --num_workers 4

/kaggle/working/TransNeXt/classification
/kaggle/working/venv/lib/python3.8/site-packages/torch/distributed/launch.py:181: FutureWarning: The module torch.distributed.launch is deprecated
and will be removed in future. Use torchrun.
Note that --use-env is set by default in torchrun.
If your script expects `--local-rank` argument to be set, please
change it to read from `os.environ['LOCAL_RANK']` instead. See 
https://pytorch.org/docs/stable/distributed.html#launch-utility for 
further instructions

  warnings.warn(
/kaggle/working/venv/lib/python3.8/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(
swattention package found, loading CUDA version

In [20]:
%cd /kaggle/working/TransNeXt/classification
!bash dist_train.sh ./configs/finetune/transnext_base_384_ft.py \
  1 \
  --data-path /kaggle/tmp/isic_2018_task3 \
  --data-set ISIC_2018 \
  --resume ./checkpoints/transnext_base_384/best.pth \
  --eval \
  --eval-split test \
  --input-size 512 \
  --fixed-pool-size 7 \
  --batch-size 32 \
  --num_workers 4

/kaggle/working/TransNeXt/classification
/kaggle/working/venv/lib/python3.8/site-packages/torch/distributed/launch.py:181: FutureWarning: The module torch.distributed.launch is deprecated
and will be removed in future. Use torchrun.
Note that --use-env is set by default in torchrun.
If your script expects `--local-rank` argument to be set, please
change it to read from `os.environ['LOCAL_RANK']` instead. See 
https://pytorch.org/docs/stable/distributed.html#launch-utility for 
further instructions

  warnings.warn(
/kaggle/working/venv/lib/python3.8/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(
swattention package found, loading CUDA version

In [24]:
%cd /kaggle/working/TransNeXt/classification
!bash dist_train.sh ./configs/finetune/transnext_base_384_ft.py \
  1 \
  --data-path /kaggle/tmp/isic_2018_task3 \
  --data-set ISIC_2018 \
  --resume ./checkpoints/transnext_base_384/best.pth \
  --eval \
  --eval-split test \
  --input-size 512 \
  --pretrain-size 384 \
  --batch-size 16 \
  --num_workers 4

/kaggle/working/TransNeXt/classification
/kaggle/working/venv/lib/python3.8/site-packages/torch/distributed/launch.py:181: FutureWarning: The module torch.distributed.launch is deprecated
and will be removed in future. Use torchrun.
Note that --use-env is set by default in torchrun.
If your script expects `--local-rank` argument to be set, please
change it to read from `os.environ['LOCAL_RANK']` instead. See 
https://pytorch.org/docs/stable/distributed.html#launch-utility for 
further instructions

  warnings.warn(
/kaggle/working/venv/lib/python3.8/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(
swattention package found, loading CUDA version

In [26]:
%cd /kaggle/working/TransNeXt/classification
!bash dist_train.sh ./configs/finetune/transnext_base_384_ft.py \
  1 \
  --data-path /kaggle/tmp/isic_2018_task3 \
  --data-set ISIC_2018 \
  --resume ./checkpoints/transnext_base_384/best.pth \
  --eval \
  --eval-split test \
  --input-size 640 \
  --pretrain-size 512 \
  --batch-size 16 \
  --num_workers 4

/kaggle/working/TransNeXt/classification
/kaggle/working/venv/lib/python3.8/site-packages/torch/distributed/launch.py:181: FutureWarning: The module torch.distributed.launch is deprecated
and will be removed in future. Use torchrun.
Note that --use-env is set by default in torchrun.
If your script expects `--local-rank` argument to be set, please
change it to read from `os.environ['LOCAL_RANK']` instead. See 
https://pytorch.org/docs/stable/distributed.html#launch-utility for 
further instructions

  warnings.warn(
/kaggle/working/venv/lib/python3.8/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(
swattention package found, loading CUDA version